# Biopython Complete Function Reference Notebook

This notebook walks through the major modules of **Biopython** (v1.88) with runnable, self-contained examples.
Where a function needs an internet connection (NCBI Entrez, BLAST, PDB downloads), the code is included but wrapped
so you can run it once you have connectivity. Everything else runs completely offline using data created inline.

**Install first:**
```bash
pip install biopython
```

## Table of Contents
1. Bio.Seq — sequence objects and operations
2. Bio.SeqRecord — annotated sequences
3. Bio.SeqIO — reading/writing sequence files
4. Bio.SeqUtils — GC content, molecular weight, protein analysis
5. Bio.Align — pairwise alignment (PairwiseAligner, replaces deprecated pairwise2)
6. Bio.AlignIO — multiple sequence alignment I/O
7. Bio.Restriction — restriction enzyme digestion
8. Bio.motifs — sequence motifs and PWMs
9. Bio.Phylo — phylogenetic trees
10. Bio.PDB — protein structure parsing
11. Bio.Entrez — NCBI database access (online)
12. Bio.Blast — BLAST searches and parsing (online)
13. Bio.Data — codon tables & IUPAC data


In [2]:
import Bio
print("Biopython version:", Bio.__version__)

Biopython version: 1.88


## 1. Bio.Seq — the `Seq` object

`Seq` is Biopython's core immutable sequence class (like a string, but biology-aware).
Covers: creation, slicing, concatenation, `complement()`, `reverse_complement()`, `transcribe()`,
`back_transcribe()`, `translate()`, `count()`, `find()`, `MutableSeq`.

In [3]:
from Bio.Seq import Seq, MutableSeq

# Creating a Seq object
dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
print("Sequence:", dna)
print("Length:", len(dna))

# Slicing and indexing behave like strings
print("First 6 bases:", dna[:6])
print("Reversed:", dna[::-1])

# Concatenation
extra = Seq("AAATTT")
print("Concatenated:", dna + extra)

# Counting and searching
print("Count of 'GG':", dna.count("GG"))
print("Index of 'ATG':", dna.find("ATG"))

# Complement / reverse complement
print("Complement:", dna.complement())
print("Reverse complement:", dna.reverse_complement())

# Transcription (DNA -> mRNA) and back-transcription
mrna = dna.transcribe()
print("mRNA:", mrna)
print("Back to DNA:", mrna.back_transcribe())

# Translation (mRNA/DNA -> protein)
protein = dna.translate()
print("Protein:", protein)
print("Protein (stop at first stop codon):", dna.translate(to_stop=True))
print("Protein using table 2 (vertebrate mito):", dna.translate(table=2))

# Case conversion
print("Upper/lower:", dna.lower(), dna.lower().upper())

# MutableSeq lets you edit sequences in place
mutable = MutableSeq(str(dna))
mutable[0] = "T"
print("Mutated sequence:", mutable)

Sequence: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
Length: 39
First 6 bases: ATGGCC
Reversed: GATAGCCCGTGGGAAAGTCGCCGGGTAATGTTACCGGTA
Concatenated: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGAAATTT
Count of 'GG': 3
Index of 'ATG': 0
Complement: TACCGGTAACATTACCCGGCGACTTTCCCACGGGCTATC
Reverse complement: CTATCGGGCACCCTTTCAGCGGCCCATTACAATGGCCAT
mRNA: AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAG
Back to DNA: ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
Protein: MAIVMGR*KGAR*
Protein (stop at first stop codon): MAIVMGR
Protein using table 2 (vertebrate mito): MAIVMGRWKGAR*
Upper/lower: atggccattgtaatgggccgctgaaagggtgcccgatag ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG
Mutated sequence: TTGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG


## 2. Bio.SeqRecord — sequences with metadata

`SeqRecord` wraps a `Seq` with an id, name, description, annotations, and features —
the object produced by `SeqIO.parse`/`SeqIO.read`.

In [4]:
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio.SeqFeature import SeqFeature, FeatureLocation

record = SeqRecord(
    Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"),
    id="EX001",
    name="ExampleGene",
    description="A toy gene for demonstration purposes",
)

# Add annotations (free-form dictionary)
record.annotations["organism"] = "Homo sapiens (example)"
record.annotations["source"] = "synthetic"

# Add a feature (e.g. a CDS spanning the whole sequence)
feature = SeqFeature(FeatureLocation(0, len(record.seq)), type="CDS", qualifiers={"gene": ["EX1"]})
record.features.append(feature)

print(record)
print("\nID:", record.id, "| Name:", record.name)
print("Description:", record.description)
print("Annotations:", record.annotations)
print("Features:", record.features)

# Slicing a SeqRecord keeps relevant features
sub_record = record[0:15]
print("\nSub-record:", sub_record.seq)

ID: EX001
Name: ExampleGene
Description: A toy gene for demonstration purposes
Number of features: 1
/organism=Homo sapiens (example)
/source=synthetic
Seq('ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG')

ID: EX001 | Name: ExampleGene
Description: A toy gene for demonstration purposes
Annotations: {'organism': 'Homo sapiens (example)', 'source': 'synthetic'}
Features: [SeqFeature(SimpleLocation(ExactPosition(0), ExactPosition(39)), type='CDS', qualifiers=...)]

Sub-record: ATGGCCATTGTAATG


## 3. Bio.SeqIO — reading and writing sequence files

Covers `SeqIO.parse()`, `SeqIO.read()`, `SeqIO.write()`, `SeqIO.convert()`, and `SeqIO.to_dict()`.
We create small FASTA/GenBank-style files on disk to demonstrate every function without needing external data.

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Build a small FASTA file with three records
records = [
    SeqRecord(Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGG"), id="seq1", description="gene A"),
    SeqRecord(Seq("TTGGCCATTGTAATGGGCCGCTGAAAGGG"), id="seq2", description="gene B"),
    SeqRecord(Seq("ATGGCCATTCTAATGGGCCGCTGAAACGG"), id="seq3", description="gene C"),
]
SeqIO.write(records, "example.fasta", "fasta")
print("Wrote example.fasta")

# SeqIO.parse -> iterate over all records (memory efficient, generator)
print("\n--- SeqIO.parse ---")
for rec in SeqIO.parse("example.fasta", "fasta"):
    print(rec.id, len(rec.seq))

# SeqIO.read -> use only when the file has exactly ONE record
single = SeqRecord(Seq("ATGGCCATTGTAATGGGCCGCTGA"), id="single1", description="only record")
SeqIO.write(single, "single.fasta", "fasta")
one_record = SeqIO.read("single.fasta", "fasta")
print("\n--- SeqIO.read ---")
print(one_record.id, one_record.seq)

# SeqIO.to_dict -> random access by id
record_dict = SeqIO.to_dict(SeqIO.parse("example.fasta", "fasta"))
print("\n--- SeqIO.to_dict ---")
print("seq2 ->", record_dict["seq2"].seq)

# SeqIO.index -> like to_dict but lazy (good for huge files)
indexed = SeqIO.index("example.fasta", "fasta")
print("\n--- SeqIO.index ---")
print("seq3 ->", indexed["seq3"].seq)
indexed.close()

# SeqIO.convert -> change file format in one call (FASTA -> Tab-delimited)
count = SeqIO.convert("example.fasta", "fasta", "example.tab", "tab")
print(f"\nConverted {count} records to example.tab")

## 4. Bio.SeqUtils — sequence statistics and protein analysis

Covers `gc_fraction`, `molecular_weight`, `six_frame_translations`, `seq1`/`seq3` (amino-acid code conversion),
and `ProtParam.ProteinAnalysis` for physicochemical properties.

In [ ]:
from Bio.SeqUtils import gc_fraction, molecular_weight, seq1, seq3, six_frame_translations
from Bio.Seq import Seq

dna = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# GC content as a fraction (0-1); multiply by 100 for percent
print("GC fraction:", gc_fraction(dna))

# Molecular weight (Da) for DNA, RNA or protein sequences
print("Molecular weight (DNA):", molecular_weight(dna))
print("Molecular weight (protein):", molecular_weight(Seq("MAIVMGR"), seq_type="protein"))

# Convert amino-acid codes: 3-letter -> 1-letter and back
print("seq1('MetAlaIleVal'):", seq1("MetAlaIleVal"))
print("seq3('MAIV'):", seq3("MAIV"))

# Six-frame translation (all 3 forward + 3 reverse reading frames) - useful for ORF hunting
print("\nSix frame translations:\n", six_frame_translations(dna))

In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

protein_seq = "MAIVMGRWKGAR"
analysed = ProteinAnalysis(protein_seq)

print("Molecular weight:", analysed.molecular_weight())
print("Amino acid percent:", analysed.get_amino_acids_percent())
print("Aromaticity:", analysed.aromaticity())
print("Instability index:", analysed.instability_index())
print("Isoelectric point:", analysed.isoelectric_point())
print("Secondary structure fraction (helix, turn, sheet):", analysed.secondary_structure_fraction())
print("Gravy (hydropathy):", analysed.gravy())

In [ ]:
from Bio.SeqUtils import MeltingTemp as mt
from Bio.Seq import Seq

primer = Seq("ATGGCCATTGTAATGGGCCGCTG")
print("Tm (Wallace rule):", mt.Tm_Wallace(primer))
print("Tm (GC method):", mt.Tm_GC(primer))
print("Tm (Nearest-Neighbor method):", mt.Tm_NN(primer))

## 5. Bio.Align — pairwise sequence alignment

`Bio.pairwise2` is **deprecated** (removed guidance since Biopython 1.80). The modern replacement is
`Bio.Align.PairwiseAligner`, which supports global/local alignment, custom scoring, and substitution matrices.

In [ ]:
from Bio import Align
from Bio.Align import substitution_matrices

seq1 = "GATTACA"
seq2 = "GCATGCU"

# --- Simple global alignment with match/mismatch/gap scores ---
aligner = Align.PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -0.5

alignments = aligner.align(seq1, seq2)
best = alignments[0]
print("Best global alignment score:", best.score)
print(best)

# --- Local alignment ---
aligner.mode = "local"
local_alignments = aligner.align(seq1, seq2)
print("Best local alignment score:", local_alignments[0].score)
print(local_alignments[0])

# --- Protein alignment using a BLOSUM62 substitution matrix ---
protein_aligner = Align.PairwiseAligner()
protein_aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
protein_aligner.open_gap_score = -10
protein_aligner.extend_gap_score = -0.5
protein_alignments = protein_aligner.align("MKVILAT", "MKVLLAT")
print("\nProtein alignment score (BLOSUM62):", protein_alignments[0].score)
print(protein_alignments[0])

# List available built-in substitution matrices
print("\nAvailable substitution matrices (first 10):", substitution_matrices.load()[:10])

## 6. Bio.AlignIO — multiple sequence alignment I/O

Covers `AlignIO.read`, `AlignIO.parse`, `AlignIO.write`, and the `MultipleSeqAlignment` object,
plus computing a quick consensus with `Bio.Align.AlignInfo`.

In [ ]:
from Bio.Align import MultipleSeqAlignment, AlignInfo
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio import AlignIO

msa = MultipleSeqAlignment([
    SeqRecord(Seq("ATGGCCATTGTA"), id="seq1"),
    SeqRecord(Seq("ATGGCCATCGTA"), id="seq2"),
    SeqRecord(Seq("ATGACCATTGTA"), id="seq3"),
])

# Write to a CLUSTAL-formatted file, then read it back
AlignIO.write(msa, "example.aln", "clustal")
alignment = AlignIO.read("example.aln", "clustal")
print(alignment)

# Column-wise slicing: get alignment columns 0-3 across all sequences
print("\nFirst 4 columns:\n", alignment[:, 0:4])

# Consensus sequence
summary = AlignInfo.SummaryInfo(alignment)
print("\nConsensus:", summary.dumb_consensus())

# Position-specific scoring matrix (PSSM)
pssm = summary.pos_specific_score_matrix()
print("\nPSSM (first rows):\n", pssm)

## 7. Bio.Restriction — restriction enzyme digestion

Simulates cutting DNA with restriction enzymes (e.g. `EcoRI`, `BamHI`) — useful for cloning/plasmid design.

In [ ]:
from Bio.Restriction import EcoRI, BamHI, Analysis, RestrictionBatch
from Bio.Seq import Seq

dna = Seq("GAATTCGGATCCATGGCCATTGTAATGGGATCCGAATTC")

# Find all cut positions for a single enzyme
print("EcoRI cut sites:", EcoRI.search(dna))
print("BamHI cut sites:", BamHI.search(dna))

# Analyse with a batch of several enzymes at once
batch = RestrictionBatch([EcoRI, BamHI])
analysis = Analysis(batch, dna)
analysis.print_that()  # pretty-prints a cut-site report to stdout

## 8. Bio.motifs — sequence motifs and position weight matrices

Build a motif from aligned binding-site sequences and compute a consensus and PWM
(e.g. for a transcription-factor binding motif).

In [ ]:
from Bio import motifs
from Bio.Seq import Seq

instances = [Seq("TACAA"), Seq("TACGC"), Seq("TACAC"), Seq("TACCC"), Seq("AACCC")]
m = motifs.create(instances)

print("Consensus:", m.consensus)
print("Anticonsensus:", m.anticonsensus)
print("Counts matrix:\n", m.counts)

pwm = m.counts.normalize(pseudocounts=0.5)
print("\nPosition weight matrix:\n", pwm)

pssm = pwm.log_odds()
print("\nLog-odds PSSM:\n", pssm)

# Scan a longer sequence for matches above a score threshold
test_seq = Seq("GGGGTACAAGGGGTACGCTTTT")
for position, score in pssm.search(test_seq, threshold=3.0):
    print(f"Match at position {position} with score {score:.2f}")

## 9. Bio.Phylo — phylogenetic trees

Read, inspect, and render trees in Newick/Nexus format. Uses a small inline Newick string
so it works fully offline.

In [ ]:
from Bio import Phylo
from io import StringIO

newick_str = "(((A:0.1,B:0.2):0.05,C:0.3):0.1,D:0.4);"
tree = Phylo.read(StringIO(newick_str), "newick")

print("Tree terminals:", [t.name for t in tree.get_terminals()])
print("Total branch length:", tree.total_branch_length())

# ASCII rendering (works in any terminal/notebook without matplotlib)
Phylo.draw_ascii(tree)

# Depth / distance between tips
print("Distance A to D:", tree.distance("A", "D"))

# Draw with matplotlib (requires matplotlib backend)
try:
    import matplotlib
    matplotlib.use("Agg")
    Phylo.draw(tree, do_show=False)
except Exception as e:
    print("matplotlib drawing skipped:", e)

## 10. Bio.PDB — protein structure parsing

Parses PDB/mmCIF structure files into a Structure/Model/Chain/Residue/Atom hierarchy.
Below we build a minimal in-memory PDB file so it runs without downloading anything;
swap in `PDBList().retrieve_pdb_file("1CRN")` to fetch a real structure when online.

In [ ]:
from Bio.PDB import PDBParser
import io

minimal_pdb = '''\
ATOM      1  N   ALA A   1      11.104  13.207   2.052  1.00 20.00           N
ATOM      2  CA  ALA A   1      12.560  13.207   2.052  1.00 20.00           C
ATOM      3  C   ALA A   1      13.100  14.620   2.052  1.00 20.00           C
ATOM      4  O   ALA A   1      12.400  15.630   2.052  1.00 20.00           O
ATOM      5  N   GLY A   2      14.430  14.670   2.052  1.00 20.00           N
END
'''

parser = PDBParser(QUIET=True)
structure = parser.get_structure("EXAMPLE", io.StringIO(minimal_pdb))

for model in structure:
    for chain in model:
        print("Chain:", chain.id)
        for residue in chain:
            print(" Residue:", residue.get_resname(), residue.id[1])
            for atom in residue:
                print("  Atom:", atom.get_name(), atom.get_coord())

# --- To fetch a real structure from the PDB (requires internet) ---
# from Bio.PDB import PDBList
# pdbl = PDBList()
# pdbl.retrieve_pdb_file("1CRN", pdir=".", file_format="pdb")

## 11. Bio.Entrez — NCBI database access (requires internet)

Always set `Entrez.email` (and optionally `Entrez.api_key`) before querying, per NCBI's usage policy.
These calls need a live internet connection; they are shown here for completeness.

In [ ]:
from Bio import Entrez, SeqIO

Entrez.email = "your.email@example.com"  # required by NCBI

# esearch: find record IDs matching a query
# handle = Entrez.esearch(db="nucleotide", term="Homo sapiens BRCA1", retmax=5)
# search_results = Entrez.read(handle)
# handle.close()
# print(search_results["IdList"])

# efetch: download a specific record by accession/GI
# with Entrez.efetch(db="nucleotide", id="NM_007294", rettype="gb", retmode="text") as handle:
#     record = SeqIO.read(handle, "genbank")
# print(record.id, record.description)

# esummary: get a lightweight summary instead of the full record
# handle = Entrez.esummary(db="nucleotide", id="NM_007294")
# summary = Entrez.read(handle)
# handle.close()

# elink: find related records in another database
# handle = Entrez.elink(dbfrom="nucleotide", db="protein", id="NM_007294")
# links = Entrez.read(handle)

print("Uncomment the calls above once you have internet access; Entrez.email must always be set.")

## 12. Bio.Blast — running and parsing BLAST (requires internet)

`Bio.Blast.qblast()` submits a search to NCBI's remote BLAST service; `Blast.parse()` reads
the returned XML into `Alignment`/`Hit` objects.

In [ ]:
from Bio import Blast
from Bio.Seq import Seq

query = Seq("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")

# Submit a remote BLAST search (blastn against the 'nt' database) - needs internet, can take minutes
# result_stream = Blast.qblast("blastn", "nt", query)
# blast_records = Blast.parse(result_stream)
# for record in blast_records:
#     for hit in record:
#         print(hit.target.id, hit.target.description)
#         for alignment in hit:
#             print("  score:", alignment.score, "length:", alignment.length)

print("Uncomment to run a live BLAST search; results typically take 30s-several minutes from NCBI.")

## 13. Bio.Data — codon tables and IUPAC data

Static reference tables: the genetic code, IUPAC ambiguity codes, and average nucleotide/protein weights.

In [ ]:
from Bio.Data import CodonTable
from Bio.Data.IUPACData import ambiguous_dna_letters, protein_letters_3to1

# Standard genetic code table (table id 1)
standard_table = CodonTable.unambiguous_dna_by_id[1]
print(standard_table)

print("\nAmbiguous DNA letters:", ambiguous_dna_letters)
print("Alanine 3-letter -> 1-letter:", protein_letters_3to1["Ala"])

# Look up which codons encode a given amino acid
forward_table = standard_table.forward_table
codons_for_met = [c for c, aa in forward_table.items() if aa == "M"]
print("Codons encoding Methionine:", codons_for_met)
print("Stop codons:", standard_table.stop_codons)
print("Start codons:", standard_table.start_codons)

## Summary

| Module | Purpose | Key functions/classes |
|---|---|---|
| `Bio.Seq` | Core sequence class | `Seq`, `MutableSeq`, `.translate()`, `.transcribe()`, `.reverse_complement()` |
| `Bio.SeqRecord` | Sequence + metadata | `SeqRecord`, `.features`, `.annotations` |
| `Bio.SeqIO` | File I/O | `parse`, `read`, `write`, `convert`, `to_dict`, `index` |
| `Bio.SeqUtils` | Sequence statistics | `gc_fraction`, `molecular_weight`, `MeltingTemp`, `ProtParam` |
| `Bio.Align` | Pairwise alignment | `PairwiseAligner`, `substitution_matrices` |
| `Bio.AlignIO` / `AlignInfo` | MSA I/O & consensus | `read`, `write`, `SummaryInfo` |
| `Bio.Restriction` | Restriction digestion | `EcoRI`, `Analysis`, `RestrictionBatch` |
| `Bio.motifs` | Motif/PWM analysis | `create`, `.counts`, `.pssm` |
| `Bio.Phylo` | Phylogenetics | `read`, `draw_ascii`, `draw`, `.distance()` |
| `Bio.PDB` | 3D structures | `PDBParser`, `PDBList` |
| `Bio.Entrez` | NCBI queries | `esearch`, `efetch`, `esummary`, `elink` |
| `Bio.Blast` | BLAST searches | `qblast`, `parse` |
| `Bio.Data` | Reference tables | `CodonTable`, `IUPACData` |

Run the cells top to bottom; the online sections (Entrez/BLAST/PDB download) are commented out
so the notebook executes cleanly offline, uncomment them when you have internet access.
